In [ ]:
# -*- coding: utf-8 -*-
"""
Comparação de compensação térmica:
 - Random Forest por frequência
 - Park (1999)
Sem combinação híbrida!
Mantém o estilo original de Luiz E. A. José
"""

import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

# Conjuntos fixos
TEMPS_TREINO = {0, 10, 40, 60, 20}
TEMPS_PROVA  = {-10, 30, 50, 70}

# Park (comparação)
PARK_MAX_SHIFT_FRAC = 0.25
PARK_OVERLAP_MIN    = 0.60
PARK_SMOOTH_WIN     = 5
SMOOTH_WIN          = 5

# ===================== FUNÇÕES AUXILIARES =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f / 1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr
    r = win // 2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s / float(win)

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

# ===================== NORMALIZAÇÃO =====================
def normalize_per_freq(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0) + 1e-9
    Xn = (X - mean) / std
    return Xn, mean, std

# ===================== PARK (1999) =====================
def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=PARK_MAX_SHIFT_FRAC,
                           overlap_min_frac=PARK_OVERLAP_MIN,
                           smooth_win=PARK_SMOOTH_WIN):
    n = len(x)
    fmin, fmax = fhz[0], fhz[-1]
    df_band = fmax - fmin
    tau_max = max_shift_frac * df_band
    nsteps = 101
    tau_vals = np.linspace(-tau_max, tau_max, nsteps)
    best = (np.inf, 0.0, 0.0)
    for tau in tau_vals:
        x_shift = shift_interp(x, fhz, tau)
        xs = x_shift; yr = y_ref
        if len(xs) < int(overlap_min_frac * n): continue
        deltaS = float(np.mean(yr - xs))
        resid = yr - (xs + deltaS)
        Va = float(np.sum(resid * resid))
        if Va < best[0]: best = (Va, tau, deltaS)
    _, tau_best, dS_best = best
    yout = shift_interp(x, fhz, tau_best) + dS_best
    if smooth_win > 1 and smooth_win % 2 == 1:
        yout = moving_average(yout, smooth_win)
    return yout, tau_best, dS_best

def park_batch(X, y_ref, fhz):
    n, m = X.shape
    Y = np.zeros_like(X)
    taus, deltas = [], []
    for i in range(n):
        yi, tau, dS = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = yi; taus.append(tau); deltas.append(dS)
    return Y, np.array(taus), np.array(deltas)

# ===================== RANDOM FOREST POR FREQUÊNCIA =====================
def fit_per_freq_rf(X, T, fhz):
    """Treina um RF para cada frequência."""
    models = {}
    T = np.asarray(T, float).reshape(-1,1)
    print("Treinando Random Forest por frequência...")
    for j, f in tqdm(enumerate(fhz), total=len(fhz)):
        y = X[:, j]
        rf = RandomForestRegressor(
            n_estimators=300, max_depth=12,
            min_samples_leaf=1, random_state=42, n_jobs=-1
        )
        rf.fit(T, y)
        models[f] = rf
    return models

def compensate_rf(models, T_eval, fhz):
    """Prediz o espectro compensado para a temperatura de referência."""
    T_eval = np.array([[T_eval]])
    Y = [float(models[f].predict(T_eval)[0]) for f in fhz]
    return np.array(Y)

# ===================== MÉTRICAS =====================
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    rmsd = np.sqrt(np.mean((y_pred - y_true)**2))
    corr = np.corrcoef(y_true, y_pred)[0,1]
    ccdm = np.mean(np.abs(corr - 1))
    return dict(R2=r2, RMSE=rmse, MAE=mae, RMSD=rmsd, CCDM=ccdm)

# ===================== CARGA DE BASE =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz)
common_cols = [common_cols[i] for i in order]
fhz = fhz[order]
fkHz = fhz / 1e3

tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()
X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# Referência a 20°C
pool_20=[]
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
y_ref = np.median(np.vstack(pool_20), axis=0)

# ===================== NORMALIZAÇÃO =====================
X_tr_n, mean_tr, std_tr = normalize_per_freq(X_tr)
X_te_n = (X_te - mean_tr) / std_tr
y_ref_n = (y_ref - mean_tr) / std_tr

# ===================== RANDOM FOREST =====================
rf_models = fit_per_freq_rf(X_tr_n, T_tr, fhz)
Y_te_rf_n = np.array([compensate_rf(rf_models, REF_TEMP, fhz) for _ in range(len(X_te_n))])
Y_te_rf = Y_te_rf_n * std_tr + mean_tr

# ===================== PARK =====================
Y_te_park, _, _ = park_batch(X_te, y_ref, fhz)

# ===================== MÉTRICAS =====================
y_true_all = np.tile(y_ref, (len(T_te), 1))
metrics_rf = compute_metrics(y_true_all.ravel(), Y_te_rf.ravel())
metrics_pk = compute_metrics(y_true_all.ravel(), Y_te_park.ravel())

print(f"\n== MÉTRICAS GLOBAIS (Faixa {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz) ==")
print(f"RF-Comp → R2={metrics_rf['R2']:.3f} | RMSE={metrics_rf['RMSE']:.3f} | MAE={metrics_rf['MAE']:.3f} | RMSD={metrics_rf['RMSD']:.3f} | CCDM={metrics_rf['CCDM']:.3f}")
print(f"Park    → R2={metrics_pk['R2']:.3f} | RMSE={metrics_pk['RMSE']:.3f} | MAE={metrics_pk['MAE']:.3f} | RMSD={metrics_pk['RMSD']:.3f} | CCDM={metrics_pk['CCDM']:.3f}")

print("\n== MÉTRICAS POR TEMPERATURA ==")
for temp_eval in sorted(set(T_te)):
    mask = (T_te == temp_eval)
    y_true = np.tile(y_ref, (mask.sum(), 1))
    m_rf = compute_metrics(y_true.ravel(), Y_te_rf[mask].ravel())
    m_pk = compute_metrics(y_true.ravel(), Y_te_park[mask].ravel())
    print(f"T={temp_eval:>4}°C → RF: R2={m_rf['R2']:.3f} | RMSE={m_rf['RMSE']:.3f} | MAE={m_rf['MAE']:.3f} "
          f"|| Park: R2={m_pk['R2']:.3f} | RMSE={m_pk['RMSE']:.3f} | MAE={m_pk['MAE']:.3f}")

# ===================== PLOTS =====================
def _prep_plot():
    plt.rcParams.update({
        "figure.figsize": (9.2, 5.0),
        "axes.grid": True, "grid.alpha": 0.3,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelsize": 12, "axes.titlesize": 13,
        "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 10, "lines.linewidth": 1.8,
    })

def plot_rf(i=0):
    _prep_plot()
    T_real = float(te_restr.iloc[i]["temp_c"])
    plt.plot(fkHz, y_ref, '--', c='black', lw=1.2, label=f"Ref {REF_TEMP}°C")
    plt.plot(fkHz, X_te[i], c='tab:red', alpha=0.6, label=f"Original {T_real:.0f}°C")
    plt.plot(fkHz, Y_te_rf[i], c='tab:blue', lw=2, label=f"RF-Comp {T_real:.0f}°C")
    plt.title(f"Random Forest — {T_real:.0f}°C | {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    plt.xlabel("Frequência (kHz)"); plt.ylabel("Magnitude normalizada")
    plt.legend(); plt.tight_layout(); plt.show()

def plot_park(i=0):
    _prep_plot()
    T_real = float(te_restr.iloc[i]["temp_c"])
    plt.plot(fkHz, y_ref, '--', c='black', lw=1.2, label=f"Ref {REF_TEMP}°C")
    plt.plot(fkHz, X_te[i], c='tab:red', alpha=0.6, label=f"Original {T_real:.0f}°C")
    plt.plot(fkHz, Y_te_park[i], c='tab:green', lw=2, label=f"Park {T_real:.0f}°C")
    plt.title(f"Park — {T_real:.0f}°C | {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    plt.xlabel("Frequência (kHz)"); plt.ylabel("Magnitude normalizada")
    plt.legend(); plt.tight_layout(); plt.show()

# Exemplos
plot_rf(i=0)
plot_park(i=0)
